In [0]:
display(dbutils.fs.ls("/Volumes/credit_risk/bronze/volume/home-credit-default-risk/"))


# CREDIT RISK INTELLIGENCE PLATFORM
# ETAPA: INGESTÃO DOS DADOS → CAMADA BRONZE



In [0]:
# ============================================================
# 1. IMPORTS
# ============================================================

# Funções do PySpark que serão utilizadas no pipeline.
from pyspark.sql import functions as F

# Biblioteca utilizada para padronização dos nomes
# dos arquivos e tabelas.
import re

In [0]:
# ============================================================
# 2. CAMINHO DOS DADOS DE ORIGEM
# ============================================================

# Diretório onde os arquivos originais da Home Credit
# foram armazenados no Volume do Databricks.
#
# Esse diretório representa nossa fonte de dados brutos.
source_path = "/Volumes/credit_risk/bronze/volume/home-credit-default-risk/"

print(f"Caminho de origem: {source_path}")

In [0]:
# ============================================================
# 3. LISTAGEM DOS ARQUIVOS
# ============================================================

# Lista todos os arquivos existentes no Volume.
files = dbutils.fs.ls(source_path)

# Exibe os arquivos encontrados.
for file in files:
    print(file.name)

In [0]:
# ============================================================
# 4. SELEÇÃO DOS ARQUIVOS PARA INGESTÃO
# ============================================================

# Seleciona somente os arquivos CSV que serão utilizados
# no nosso pipeline de dados.
#
# O sample_submission.csv não será utilizado porque
# é um arquivo auxiliar da competição.
#
# O HomeCredit_columns_description.csv também não será
# transformado em tabela Bronze neste momento, pois
# funciona como dicionário de dados.

csv_files = [
    file.path
    for file in files
    if file.name.endswith(".csv")
    and file.name != "sample_submission.csv"
    and file.name != "HomeCredit_columns_description.csv"
]

# Exibe os arquivos selecionados.
for path in csv_files:
    print(path)

In [0]:
# ============================================================
# 5. FUNÇÃO DE INGESTÃO
# ============================================================

def ingest_to_bronze(path):
    """
    Lê um arquivo CSV do Volume e cria uma tabela Delta
    na camada Bronze.
    """

    # --------------------------------------------------------
    # Obtém o nome do arquivo.
    # --------------------------------------------------------

    # Exemplo:
    # application_train.csv
    file_name = path.split("/")[-1]

    
    # --------------------------------------------------------
    # Remove a extensão .csv.
    # --------------------------------------------------------

    # Exemplo:
    # application_train.csv
    #        ↓
    # application_train
    table_name = file_name.replace(".csv", "")

    
    # --------------------------------------------------------
    # Padroniza o nome da tabela.
    # --------------------------------------------------------

    # Substitui caracteres especiais por "_".
    table_name = re.sub(
        r"[^a-zA-Z0-9_]",
        "_",
        table_name
    )

    
    # Converte o nome para letras minúsculas.
    table_name = table_name.lower()

    
    # --------------------------------------------------------
    # Exibe o arquivo que está sendo processado.
    # --------------------------------------------------------

    print(f"Processando: {file_name}")


    # --------------------------------------------------------
    # Leitura do arquivo CSV.
    # --------------------------------------------------------

    # header=True:
    # primeira linha contém os nomes das colunas.
    #
    # inferSchema=True:
    # Spark tenta identificar automaticamente os tipos.
    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(path)
    )


    # --------------------------------------------------------
    # Adiciona timestamp de ingestão.
    # --------------------------------------------------------

    # Registra quando o registro foi processado.
    df = df.withColumn(
        "_ingestion_timestamp",
        F.current_timestamp()
    )


    # --------------------------------------------------------
    # Adiciona a origem do arquivo.
    # --------------------------------------------------------

    # Permite rastrear de qual arquivo o registro veio.
    df = df.withColumn(
        "_source_file",
        F.lit(path)
    )


    # --------------------------------------------------------
    # Grava como tabela Delta na Bronze.
    # --------------------------------------------------------

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(
            f"credit_risk.bronze.{table_name}"
        )
    )


    # --------------------------------------------------------
    # Mensagem de conclusão.
    # --------------------------------------------------------

    print(
        f"✓ Criada: "
        f"credit_risk.bronze.{table_name}"
    )

In [0]:
# ============================================================
# 6. EXECUÇÃO DA INGESTÃO
# ============================================================

# Percorre todos os arquivos selecionados
# e executa a função de ingestão.
for path in csv_files:
    ingest_to_bronze(path)

In [0]:
%sql
-- ============================================================
-- 7. VALIDAÇÃO DAS TABELAS BRONZE
-- ============================================================

-- Lista todas as tabelas existentes no schema Bronze.

SHOW TABLES IN credit_risk.bronze;

In [0]:
%sql
-- ============================================================
-- VALIDAÇÃO DA APPLICATION_TRAIN
-- ============================================================

SELECT
    COUNT(*) AS total_registros
FROM credit_risk.bronze.application_train;

In [0]:
%sql
-- Visualiza alguns registros da tabela Bronze.

SELECT *
FROM credit_risk.bronze.application_train
LIMIT 10;